<a href="https://colab.research.google.com/github/Ethan-Brooke/APF-Paper-4-Admissibility-Constraints-Field-Content/blob/main/APF_Reviewer_Walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 4 — Admissibility Constraints and Structural Saturation
## Reviewer Walkthrough — Interactive Mathematical Derivation

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18604845.svg)](https://doi.org/10.5281/zenodo.18604845) [![GitHub](https://img.shields.io/badge/GitHub-APF--Paper--4--Admissibility--Constraints--Field--Content-blue?logo=github)](https://github.com/Ethan-Brooke/APF-Paper-4-Admissibility-Constraints-Field-Content)

This notebook is a **self-contained verification** of every theorem in Paper 4 — Admissibility Constraints and Structural Saturation. Each theorem gets a markdown cell (formula, proof sketch, dependencies) followed by a live code cell that executes the check and unpacks the mathematical result.

**The claim:** the 40 theorems below are all [P]-derived from Axiom A1 (finite information capacity) plus PLEC's structural components. No free parameters are introduced at any step.

**To run end-to-end:** *Runtime → Run all* (or `Cmd/Ctrl + F9`). Total verification takes under a minute.

**Codebase:** APF v6.9 — 355 `verify_all` checks across 342 bank-registered theorems in 19 modules, 48 quantitative predictions, **0 free parameters**. This repo bundles the **40-check subset** directly cited by Paper 4.

---

### The APF corpus

Paper 4 is one companion in a 9-paper series. The full set:

| # | Title | Zenodo DOI | GitHub repo | Status |
|---|---|---|---|---|
| 0 | What Physics Permits | [10.5281/zenodo.18605692](https://doi.org/10.5281/zenodo.18605692) | [`APF-Paper-0-What-Physics-Permits`](https://github.com/Ethan-Brooke/APF-Paper-0-What-Physics-Permits) | public |
| 1 | The Enforceability of Distinction | [10.5281/zenodo.18604678](https://doi.org/10.5281/zenodo.18604678) | [`APF-Paper-1-The-Enforceability-of-Distinction`](https://github.com/Ethan-Brooke/APF-Paper-1-The-Enforceability-of-Distinction) | public |
| 2 | The Structure of Admissible Physics | [10.5281/zenodo.18604839](https://doi.org/10.5281/zenodo.18604839) | [`APF-Paper-2-The-Structure-of-Admissible-Physics`](https://github.com/Ethan-Brooke/APF-Paper-2-The-Structure-of-Admissible-Physics) | public |
| 3 | Ledgers | [10.5281/zenodo.18604844](https://doi.org/10.5281/zenodo.18604844) | [`APF-Paper-3-Ledgers-Entropy-Time-Cost`](https://github.com/Ethan-Brooke/APF-Paper-3-Ledgers-Entropy-Time-Cost) | public |
| 4 | Admissibility Constraints and Structural Saturation **(this repo)** | [10.5281/zenodo.18604845](https://doi.org/10.5281/zenodo.18604845) | [`APF-Paper-4-Admissibility-Constraints-Field-Content`](https://github.com/Ethan-Brooke/APF-Paper-4-Admissibility-Constraints-Field-Content) | public |
| 5 | Quantum Structure from Finite Enforceability | [10.5281/zenodo.18604861](https://doi.org/10.5281/zenodo.18604861) | [`APF-Paper-5-Quantum-Structure-Hilbert-Born`](https://github.com/Ethan-Brooke/APF-Paper-5-Quantum-Structure-Hilbert-Born) | public |
| 6 | Dynamics and Geometry as Optimal Admissible Reallocation | [10.5281/zenodo.18604874](https://doi.org/10.5281/zenodo.18604874) | [`APF-Paper-6-Dynamics-Geometry-Spacetime-Gravity`](https://github.com/Ethan-Brooke/APF-Paper-6-Dynamics-Geometry-Spacetime-Gravity) | public |
| 7 | Action, Internalization, and the Lagrangian | [10.5281/zenodo.18604875](https://doi.org/10.5281/zenodo.18604875) | [`APF-Paper-7-Action-Internalization-Lagrangian`](https://github.com/Ethan-Brooke/APF-Paper-7-Action-Internalization-Lagrangian) | public |
| 13 | The Minimal Admissibility Core | [10.5281/zenodo.18614663](https://doi.org/10.5281/zenodo.18614663) | [`APF-Paper-13-The-Minimal-Admissibility-Core`](https://github.com/Ethan-Brooke/APF-Paper-13-The-Minimal-Admissibility-Core) | public |
| — | Canonical codebase (v6.9) | [10.5281/zenodo.18604548](https://doi.org/10.5281/zenodo.18604548) | [`APF-Codebase`](https://github.com/Ethan-Brooke/APF-Codebase) | pending |

The canonical engine (full 342-theorem bank) is at [APF-Codebase](https://doi.org/10.5281/zenodo.18604548).


## Setup

Clone the paper-companion repo and install the bundled `apf/` package. This gives us access to the registry of check functions and the utility helpers for exact rational arithmetic.

In [ ]:
# Clone the paper-companion repo (skip if already cloned in this runtime)
!git clone -q https://github.com/Ethan-Brooke/APF-Paper-4-Admissibility-Constraints-Field-Content.git 2>/dev/null || true
import os
if os.path.isdir('APF-Paper-4-Admissibility-Constraints-Field-Content'):
    os.chdir('APF-Paper-4-Admissibility-Constraints-Field-Content')
!pip install -e . --quiet 2>&1 | tail -1

In [ ]:
from apf.bank import REGISTRY, get_check, run_all
from fractions import Fraction
import inspect
import json

print(f'Loaded {len(REGISTRY)} checks from this repo (expected 40).')
print('Exact rational arithmetic is available via `from fractions import Fraction`.')
print('Pretty-print helper defined below for consistent result display.')

In [ ]:
# Pretty-print helper: surfaces ALL mathematical content from a result dict,
# not just a pass/fail flag.
def show(check_name):
    """Run a check and surface its full mathematical result."""
    fn = get_check(check_name)
    r = fn()
    print(f'━━━ {check_name} ━━━')
    if not isinstance(r, dict):
        print(f'  result: {r}'); return r
    status = 'PASS' if r.get('passed', True) else 'FAIL'
    print(f'  status: {status}')
    # Surface mathematical content in a stable order
    for k in ('key_result', 'result', 'value', 'theorem', 'witnesses',
              'intermediate', 'derivation', 'formula', 'predicted',
              'observed', 'error_sigma', 'depends_on', 'module'):
        if k in r:
            val = r[k]
            s = str(val)
            if len(s) > 200: s = s[:197] + '...'
            print(f'  {k}: {s}')
    # Surface anything else not already covered
    seen = {'passed', 'key_result', 'result', 'value', 'theorem',
            'witnesses', 'intermediate', 'derivation', 'formula',
            'predicted', 'observed', 'error_sigma', 'depends_on', 'module'}
    for k, v in r.items():
        if k in seen: continue
        s = str(v)
        if len(s) > 200: s = s[:197] + '...'
        print(f'  {k}: {s}')
    return r

def show_source(check_name, max_lines=30):
    """Show the first N lines of a check function's source (for transparency)."""
    fn = get_check(check_name)
    try:
        src = inspect.getsource(fn)
    except OSError:
        print('  (source unavailable)')
        return
    lines = src.split('\n')[:max_lines]
    for ln in lines:
        print(ln)
    if len(src.split('\n')) > max_lines:
        print(f'    ... ({len(src.split(chr(10)))-max_lines} more lines)')

print('Helpers ready: show(<name>), show_source(<name>).')

## What this paper claims

Field content and fermion spectrum of the Standard Model: 45 fermions in three generations, the 1-of-4680 admissible template, mass-ratio structure, Cabibbo angle, CKM matrix structure. Currently PDF-only; .tex source pending.

What follows is the theorem-by-theorem derivation, grouped by tier (axioms → lemmas → theorems → predictions). Every cell you run is a live check from the canonical bank.

---

## Tier 1 · Structural Lemmas

The four structural lemmas that bridge axioms to theorems. Each is proved directly from A1 + PLEC components; nothing external is imported.


### 1. `check_L_gauge_template_uniqueness` — L_gauge_template_uniqueness

**Statement (from codebase):** L_gauge_template_uniqueness: SU(N_c)×SU(2)×U(1) is the Unique Gauge Template.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_gauge_template_uniqueness')

### 2. `check_L_count` — L_count

**Statement (from codebase):** L_count: Capacity Counting ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â 1 structural enforcement channel = 1 unit.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_count')

### 3. `check_L_nc` — L_nc

**Statement (from codebase):** L_nc: Non-Closure from Admissibility Physics + Locality.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_nc')

### 4. `check_L_area_scaling` — L_area_scaling

**Statement (from codebase):** 

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_area_scaling')

### 5. `check_L_BH_page_curve_capacity` — L_BH_page_curve_capacity

**Statement (from codebase):** L_BH_page_curve_capacity: Page Curve from Capacity Counting [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_BH_page_curve_capacity')

### 6. `check_L_anomaly_free` — L_anomaly_free

**Statement (from codebase):** L_anomaly_free: Gauge Anomaly Cancellation Cross-Check [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_anomaly_free')

### 7. `check_L_Witten_parity` — L_Witten_parity

**Statement (from codebase):** 

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_Witten_parity')

### 8. `check_L_saturation_partition` — L_saturation_partition

**Statement (from codebase):** L_saturation_partition: Type-Count Partition is Saturation-Independent [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_saturation_partition')

### 9. `check_L_FN_ladder_uniqueness` — L_FN_ladder_uniqueness

**Statement (from codebase):** L_FN_ladder_uniqueness: q_B = (7,4,0) is Unique Cost-Minimal Partition [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_FN_ladder_uniqueness')

### 10. `check_L_capacity_per_dimension` — L_capacity_per_dimension

**Statement (from codebase):** L_capacity_per_dimension: Neutrino d_1 = x^(q_B1/d_Y) [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_capacity_per_dimension')

### 11. `check_L_Yukawa_bilinear` — L_Yukawa_bilinear

**Statement (from codebase):** L_Yukawa_bilinear: Yukawa Coupling Is Bilinear in Generation Amplitudes [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_Yukawa_bilinear')

### 12. `check_L_mass_from_capacity` — L_mass_from_capacity

**Statement (from codebase):** L_mass_from_capacity: Complete Mass Matrix Derivation — Zero FN Imports [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_mass_from_capacity')

### 13. `check_L_c_FN_gap` — L_c_FN_gap

**Statement (from codebase):** L_c_FN_gap: NNLO Coefficient c = x^Δq from FN Charge Gap [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_c_FN_gap')

### 14. `check_L_gen_path` — L_gen_path

**Statement (from codebase):** L_gen_path: Generation Graph Is a Path [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_gen_path')

### 15. `check_L_CP_channel` — L_CP_channel

**Statement (from codebase):** L_CP_channel: Channel Asymmetry Enables CP Violation [P | L_H_curv, T_q_Higgs].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_CP_channel')

### 16. `check_L_dm2_hierarchy` — L_dm2_hierarchy

**Statement (from codebase):** L_dm2_hierarchy: Neutrino Mass-Squared Splitting Ratio [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_dm2_hierarchy')

### 17. `check_L_mbb_prediction` — L_mbb_prediction

**Statement (from codebase):** L_mbb_prediction: Neutrinoless Double Beta Decay Effective Mass [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_mbb_prediction')

### 18. `check_L_DUNE_response` — L_DUNE_response

**Statement (from codebase):** L_DUNE_response: APF δ_PMNS Prediction vs DUNE/Hyper-K Sensitivity [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_DUNE_response')

### 19. `check_L_cosmological_constant` — L_cosmological_constant

**Statement (from codebase):** 

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_cosmological_constant')

---

## Tier 2 · Derived Theorems

The paper's main theorems. Each depends on lemmas and/or earlier theorems as noted in the `depends_on` chain. Epistemic tags: **[P]** = internally derived; **[P]+[I]** = APF prepares the admissible class, then an external mathematical theorem (Solèr, Gleason, HKM, Malament, Lovelock) closes the classification.


### 20. `check_T_M` — T_M

**Statement (from codebase):** T_M: Interface Monogamy.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_M')

### 21. `check_T_BH_information` — T_BH_information

**Statement (from codebase):** T_BH_information: Black Hole Information Preservation [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_BH_information')

### 22. `check_T_gauge` — T_gauge

**Statement (from codebase):** T_gauge: SU(3)*SU(2)*U(1) from Capacity Budget.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_gauge')

### 23. `check_T_field` — T_field

**Statement (from codebase):** T_field: SM Fermion Content -- Exhaustive Derivation.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_field')

### 24. `check_T_capacity_ladder` — T_capacity_ladder

**Statement (from codebase):** T_capacity_ladder: Capacity Charges from Budget [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_capacity_ladder')

### 25. `check_T_sin2theta` — T_sin2theta

**Statement (from codebase):** T_sin2theta: Weinberg Angle -- structurally derived from fixed point.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_sin2theta')

### 26. `check_T_mass_ratios` — T_mass_ratios

**Statement (from codebase):** T_mass_ratios: Six Charged Fermion Mass Ratios from Zero Parameters [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_mass_ratios')

### 27. `check_T_CKM` — T_CKM

**Statement (from codebase):** T_CKM: Zero-Parameter CKM Matrix Prediction [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_CKM')

### 28. `check_T_PMNS` — T_PMNS

**Statement (from codebase):** T_PMNS: Zero-Parameter PMNS Neutrino Mixing Matrix [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_PMNS')

### 29. `check_T_PMNS_CP` — T_PMNS_CP

**Statement (from codebase):** T_PMNS_CP: Leptonic CP Violation Vanishes Exactly [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_PMNS_CP')

### 30. `check_T_nu_ordering` — T_nu_ordering

**Statement (from codebase):** T_nu_ordering: Normal Neutrino Mass Ordering [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_nu_ordering')

---

## Supporting Results

Supporting results that don't fit the tier scheme cleanly — utility identities, red-team checks, internal consistency tests.


### 31. `name` — name

**Statement (from codebase):** 

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('name')

### 32. `check_Regime_R` — Regime_R

**Statement (from codebase):** Regime_R: PLEC Well-Posedness under R1..R4 [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_Regime_R')

### 33. `check_Theorem_R` — Theorem_R

**Statement (from codebase):** Theorem_R: Representation Requirements from Admissibility.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_Theorem_R')

### 34. `check_Regime_exit_Type_IV` — Regime_exit_Type_IV

**Statement (from codebase):** Regime_exit_Type_IV: Loss of Smooth or Local Structure [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_Regime_exit_Type_IV')

### 35. `check_T4F` — T4F

**Statement (from codebase):** T4F: Flavor-Capacity Saturation.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T4F')

### 36. `check_T7` — T7

**Statement (from codebase):** T7: Generation Bound N_gen = 3 [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T7')

### 37. `check_T25a` — T25a

**Statement (from codebase):** T25a: Overlap Bounds from Interface Monogamy.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T25a')

### 38. `check_T25b` — T25b

**Statement (from codebase):** T25b: Overlap Bound from Saturation.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T25b')

### 39. `check_T27d` — T27d

**Statement (from codebase):** T27d: gamma_2/gamma_1 = d + 1/d from Representation Principles.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T27d')

### 40. `check_T23` — T23

**Statement (from codebase):** T23: Fixed-Point Formula for sin^2theta_W.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T23')

---

## Full scorecard

Aggregate verdict across all 40 bundled checks. If all pass, the claims in Paper 4 are computationally verified against the canonical APF bank at codebase v6.9.

For the full 342-theorem bank, clone the canonical engine ([https://doi.org/10.5281/zenodo.18604548](https://doi.org/10.5281/zenodo.18604548)).

In [ ]:
results = run_all(verbose=False)
passed = sum(1 for r in results if r.get('passed', True))
failed = len(results) - passed
print(f'━━━ Paper 4 scorecard ━━━')
print(f'Bundled checks: {len(results)}')
print(f'Passed:         {passed}')
print(f'Failed:         {failed}')
if failed:
    print()
    print('FAILED:')
    for r in results:
        if not r.get('passed', True):
            print(f"  - {r.get('name', '?')}: {r.get('error', r.get('reason', '?'))}")
else:
    print()
    print('All checks pass. This is not proof of the framework — it is proof')
    print('that the claims of Paper {0} are internally consistent with the'.format(4))
    print('canonical APF computational engine. The framework itself is')
    print('defended via audit, not assertion. See ai_context/AUDIT_DISCIPLINE.md.')

---

## Next steps

- **Full derivation DAG:** [interactive visualization](https://ethan-brooke.github.io/APF-Paper-4-Admissibility-Constraints-Field-Content/) (hover theorems for dependencies, click to focus subgraphs).
- **Reviewers' guide:** [`REVIEWERS_GUIDE.md`](REVIEWERS_GUIDE.md) — physics-first 9-section walkthrough.
- **AI onboarding:** [`START_HERE.md`](START_HERE.md) — operational checklist.
- **Machine-readable repo map:** [`ai_context/repo_map.json`](ai_context/repo_map.json) — every file with purpose, audience, references.
- **Theorem catalog:** [`ai_context/theorems.json`](ai_context/theorems.json) — 342-entry bank with tags and dependencies.
- **The paper:** `.pdf` in this repo.
- **Cite:** [10.5281/zenodo.18604845](https://doi.org/10.5281/zenodo.18604845).

---

*Notebook generated by the APF `create-repo` skill. Codebase: v6.9 (355 `verify_all` checks, 342 bank-registered theorems across 19 modules, 48 quantitative predictions, 0 free parameters).*